In [5]:

%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from vector_store_manager import VectorStoreManager

In [7]:
# 1. Load data from Task 1
df = pd.read_csv('../data/processed/filtered_complaints.csv')

In [8]:
df.head()

,Date received,Product,Sub-product,Consumer complaint narrative,Complaint ID,cleaned_narrative,word_count
0,2025-06-13,Credit card,Store credit card,A XXXX XXXX card was opened under my name by a...,14069121,a xxxx xxxx card was opened under my name by a...,91
1,2025-06-13,Savings account,Checking account,I made the mistake of using my wellsfargo debi...,14061897,i made the mistake of using my wellsfargo debi...,108
2,2025-06-12,Credit card,General-purpose credit card or charge card,"Dear CFPB, I have a secured credit card with c...",14047085,"dear cfpb, i have a secured credit card with c...",156
3,2025-06-12,Credit card,General-purpose credit card or charge card,I have a Citi rewards cards. The credit balanc...,14040217,i have a citi rewards cards. the credit balanc...,231
4,2025-06-09,Credit card,General-purpose credit card or charge card,b'I am writing to dispute the following charge...,13968411,bi am writing to dispute the following charges...,452


In [4]:
# 2. Initialize Manager
vm = VectorStoreManager()

In [9]:
# 3. Step 1: Stratified Sampling (10,000 - 15,000)
sampled_df = vm.get_stratified_sample(df, sample_size=12000)

Sampling 12000 records proportionally...


e:\DS_Courses\KAIM_10_Academy\KAIM_8\Week_7\KAIM_Week7\src\vector_store_manager.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('Product', group_keys=False).apply(


Sampled distribution:
Product
Credit card        4441
Savings account    3496
Money transfers    2223
Personal loan      1840
Name: count, dtype: int64


In [10]:
# 4. Step 2 & 3: Chunking, Embedding, and Indexing
# This creates the vector_store/ folder automatically
vector_db = vm.create_vector_store(sampled_df)

Converting narratives to LangChain Documents...
Chunking 12000 documents...
Created 20888 chunks.
Indexing to ../vector_store/chroma_db (this may take a few minutes)...
Vector store successfully built and persisted.


In [ ]:
# 5. Quick Test: Semantic Search
query = "I am having trouble with a money transfer to Europe"
results = vector_db.similarity_search(query, k=3)
for i, doc in enumerate(results):
    print(f"\nResult {i+1}:")     
    print(f"Product: {doc.metadata['Product']}")
    print(f"Text Snippet: {doc.page_content[:200]}...")


Result 1:
Product: Money transfers
Text Snippet: hi, i wanted to transfer money to xxxx to help my sister whos business is severely impacted by covid shutdown. i used xxxx continental exchange solutions , inc. dba xe usa to initiate the transfer. th...

Result 2:
Product: Money transfers
Text Snippet: hi, i wanted to transfer money to xxxx to help my sister whos business is severely impacted by covid shutdown. i used xxxx continental exchange solutions , inc. dba xe usa to initiate the transfer. th...

Result 3:
Product: Money transfers
Text Snippet: taken the week of the transfer. i believe this may have something to do with the problem causing the funds to have never been received. incidentally, xxxx does receive international transfers. i have ...
